**Conditional Chains**


In [12]:
from langchain_core.prompts import ChatPromptTemplate
from langchain.chat_models import init_chat_model
from langchain_core.output_parsers import StrOutputParser, PydanticOutputParser
from langchain_core.runnables import RunnableLambda, RunnableParallel, RunnableBranch
from pydantic import BaseModel, Field
from typing import List

In [10]:
from typing import Literal

class llm_schema(BaseModel):
    audience_flag: Literal["positive", "negative"] = Field(..., description="The sentiment of the movie review")

In [14]:
#Task-1
prompt_template = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant"),
    ("user", "{user_movie_review}. Please provide a sentiment analysis of the movie review and classify it as either 'positive' or 'negative'.")
])

#Task-2
llm_openai = init_chat_model(model="gpt-5-mini", model_provider="openai", temperature=0.1)
structured_llm_output = llm_openai.with_structured_output(llm_schema)

# #Task-3
def review_flag_generator(obj: llm_schema) -> str:
    return obj.model_dump()['audience_flag']

review_flag_creator = RunnableLambda(review_flag_generator)

In [15]:
#Conditional Chain Option 1

def instagram_post_creator(text: str):
    prompt_template = ChatPromptTemplate.from_messages([
        ("system", "You are a social media expert"),   
        ("human", "Create an Instagram post on topic: {text}. Please provide a catchy caption and relevant hashtags.")
    ])

    llm_openai = init_chat_model(model="gpt-5-mini", model_provider="openai", temperature=0.1)

    str_parser = StrOutputParser()

    instagram_post = prompt_template | llm_openai | str_parser
    instagram_post_output = instagram_post.invoke({"text": text})
    return instagram_post_output

insta_chain_runnable = RunnableLambda(instagram_post_creator)

In [16]:
#Conditional Chain Option 2

def twitter_post_creator(text: str):
    prompt_template = ChatPromptTemplate.from_messages([
        ("system", "You are a social media expert"),   
        ("human", "Create a Twitter post on topic: {text}. Please provide a catchy caption and relevant hashtags.")
    ])

    llm_openai = init_chat_model(model="gpt-5-mini", model_provider="openai", temperature=0.1)

    str_parser = StrOutputParser()

    twitter_post = prompt_template | llm_openai | str_parser
    twitter_post_output = twitter_post.invoke({"text": text})
    return twitter_post_output

twitter_chain_runnable = RunnableLambda(twitter_post_creator)

**Conditional Chain**

In [20]:
final_chain = prompt_template | structured_llm_output | review_flag_creator | RunnableBranch(
    (lambda x: "positive" in x, insta_chain_runnable),
    twitter_chain_runnable
)

final_chain.invoke({"user_movie_review": "tamil movie 'gilli' is cheap copy of telugu classic okkadu! please watch original movie instead of this cheap copy. I am very disappointed with the movie. The acting is poor and the story is predictable."})

'Caption:\n"Negative thoughts will try to take the mic — don’t hand it over. Notice them, name them, then choose a better story."\n\nHashtags:\n#Negativity #MindsetMatters #MentalHealth #SelfCare #PositiveShift #ChooseJoy #Boundaries'